# 13 · Profile: National Career Clusters crosswalk (Advance CTE)

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/career_clusters.md`.

**Why this source matters.** brief S6 makes the National Career Clusters Framework the umbrella over jobs, majors and industries: 14 clusters, each split into sub-clusters, so a visitor can start broad ("Digital Technology") and drill down to one job or one major. brief S7 picks the first featured areas from these clusters (Tech, Healthcare, Business, Education). Advance CTE published the modernized framework in October 2024; O*NET adopted it in April 2025. The crosswalk files tie each cluster to SOC jobs, CIP majors and NAICS sectors.

**What this notebook does:** reads the four crosswalk workbooks, checks grain, codes and whether the files agree with each other, measures how much of NYC's jobs, projected openings and 2024 degrees they cover, and looks at how a major gets its clusters. It changes nothing.

In [1]:
import sys, zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 90, "display.width", 220)
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key == "career_clusters"][["key", "file", "downloaded_utc", "bytes", "note"]]
display(FILES)
P = {Path(f).stem.replace("Full_Framework_Crosswalk", "") or "_full": RAW / f for f in FILES.file}
print(list(P))

,key,file,downloaded_utc,bytes,note
197,career_clusters,skills/career_clusters/Full_Framework_Crosswalk.xlsx,2026-10-13T04:12:48+00:00,398659,"Career Clusters: combined CIP-SOC-NAICS -> cluster/sub-cluster (Advance CTE, modernize..."
198,career_clusters,skills/career_clusters/Full_Framework_Crosswalk_CIP-Career-Clusters.xlsx,2026-10-13T04:12:49+00:00,348485,"Career Clusters: CIP -> cluster/sub-cluster (Advance CTE, modernized framework, Sep 20..."
199,career_clusters,skills/career_clusters/Full_Framework_Crosswalk_SOC-Career-Clusters.xlsx,2026-10-13T04:12:50+00:00,198548,"Career Clusters: SOC -> cluster/sub-cluster (Advance CTE, modernized framework, Sep 20..."
200,career_clusters,skills/career_clusters/Full_Framework_Crosswalk_NAICS-Career-Clusters-Subclusters.xlsx,2026-10-13T04:12:51+00:00,190653,"Career Clusters: NAICS -> cluster/sub-cluster (Advance CTE, modernized framework, Sep ..."


['_full', '_CIP-Career-Clusters', '_SOC-Career-Clusters', '_NAICS-Career-Clusters-Subclusters']


## 0. Source header

| | **National Career Clusters® Framework crosswalks** (`career_clusters`) |
|---|---|
| **Publisher** | Advance CTE (the national association of state CTE directors), with the modernized framework released October 2024 |
| **Files** | Four workbooks from [careertech.org/resource/framework-crosswalk](https://careertech.org/resource/framework-crosswalk/), uploaded September 2025: the combined SOC – CIP – NAICS file, and one each for CIP, SOC and NAICS → cluster / sub-cluster |
| **Release** | Downloaded 2026-10-13. No version number in the files; the upload folder says 2025/09 |
| **How it was made** | **Expert assignment.** Advance CTE placed SOC 2018 jobs into sub-clusters, then brought in CIP 2020 programs through the NCES CIP ↔ SOC crosswalk and NAICS sectors by hand. Not a survey; no counts |
| **License** | Free to download and use; "National Career Clusters" is a registered trademark of Advance CTE (credit it by name) |
| **Documentation** | Each workbook's *Overview* sheet; [Framework guidebook](https://careertech.org/resource/guidebook-modernized-national-career-clusters-framework/) |

## 1. Load as text

Every sheet, read with no header so nothing is lost; then the data sheets with their header row. Codes stay text (CIP `01.0000` would lose its zero as a number).

In [2]:
for k, f in P.items():
    for name, d in pd.read_excel(f, sheet_name=None, header=None, dtype=str).items():
        print(f"{k:<36} | {name:<28} | {d.shape[0]:>5} rows x {d.shape[1]:>2} cols")
rx = lambda k, s: pd.read_excel(P[k], sheet_name=s, dtype=str).apply(lambda c: c.str.strip())
full = rx("_full", "SOC - CIP - CC")
soc = rx("_SOC-Career-Clusters", "SOC - CC - Sub-Clusters")
cip = rx("_CIP-Career-Clusters", "CIP - CC - Sub-Clusters")
nai = rx("_NAICS-Career-Clusters-Subclusters", "CC - Sub-Cluster - NAICS")
socw = rx("_SOC-Career-Clusters", "SOC - CC (Wide)")
for n, d in {"full": full, "soc": soc, "cip": cip, "naics": nai}.items():
    print(f"\n{n}: {d.shape} | exact duplicate rows: {d.duplicated().sum()} | blanks: {d.isna().sum()[d.isna().sum() > 0].to_dict()}")
display(full.head(4)); display(soc.head(3)); display(cip.head(3)); display(nai.head(3))

_full                                | Overview                     |    35 rows x  7 cols
_full                                | SOC - CIP - CC               |  6689 rows x  8 cols


_CIP-Career-Clusters                 | Overview                     |    21 rows x  7 cols
_CIP-Career-Clusters                 | CIP - CC - Sub-Clusters      |  4726 rows x  4 cols
_CIP-Career-Clusters                 | CIP - CC (Wide)              |  1947 rows x 16 cols
_CIP-Career-Clusters                 | Single Career Cluster CIPs   |   747 rows x  4 cols
_CIP-Career-Clusters                 | Interdisciplinary CIPs       |  1201 rows x 16 cols
_SOC-Career-Clusters                 | Overview                     |    21 rows x  7 cols
_SOC-Career-Clusters                 | SOC - CC - Sub-Clusters      |  1204 rows x  4 cols
_SOC-Career-Clusters                 | SOC - CC (Wide)              |   852 rows x 11 cols
_SOC-Career-Clusters                 | Single Career Cluster SOCs   |   727 rows x  4 cols
_SOC-Career-Clusters                 | Interdisciplinary SOCs       |   126 rows x 11 cols


_NAICS-Career-Clusters-Subclusters   | Overview                     |    32 rows x  7 cols
_NAICS-Career-Clusters-Subclusters   | CC - Sub-Cluster - NAICS     |   113 rows x  4 cols



full: (6688, 8) | exact duplicate rows: 2 | blanks: {'Sub-Cluster': 14, 'SOC Code': 14, 'Occupation Title': 14, 'CIP Code (2020)': 14, 'CIP Title (2020)': 14, '2-Digit NAICS': 14, 'NAICS title': 14}

soc: (1203, 4) | exact duplicate rows: 0 | blanks: {}

cip: (4725, 4) | exact duplicate rows: 0 | blanks: {}

naics: (112, 4) | exact duplicate rows: 0 | blanks: {}


,Career Cluster,Sub-Cluster,SOC Code,Occupation Title,CIP Code (2020),CIP Title (2020),2-Digit NAICS,NAICS title
0,Advanced Manufacturing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Advanced Manufacturing,Production & Automation,11-3051,Industrial Production Managers,14.3501,Industrial Engineering.,31-33,Manufacturing
2,Advanced Manufacturing,Production & Automation,11-3051,Industrial Production Managers,15.1501,Engineering/Industrial Management.,31-33,Manufacturing
3,Advanced Manufacturing,Production & Automation,11-3051,Industrial Production Managers,51.2006,Clinical and Industrial Drug Development.,31-33,Manufacturing


,SOC Code,Occupation Title,Career Cluster,Sub-Cluster
0,11-1011,Chief Executives,Management & Entrepreneurship,Leadership & Operations
1,11-1011,Chief Executives,Management & Entrepreneurship,Entrepreneurship & Small Business
2,11-1021,General and Operations Managers,Energy & Natural Resources,Conservation & Land Management


,CIP Code (2020),CIP Title (2020),Career Cluster,Sub-Cluster
0,01.0000,"Agriculture, General.",Agriculture,Animal Systems
1,01.0000,"Agriculture, General.",Agriculture,Food Science & Processing
2,01.0000,"Agriculture, General.",Agriculture,Plant Systems


,Career Cluster,Sub-Cluster,2-Digit NAICS,NAICS title
0,Advanced Manufacturing,Production & Automation,31-33,Manufacturing
1,Advanced Manufacturing,Engineering,31-33,Manufacturing
2,Advanced Manufacturing,Safety & Quality Assurance,31-33,Manufacturing


**Load findings**
- Four workbooks, each with an *Overview* sheet plus data sheets. The long sheets are the ones to use: `SOC - CC - Sub-Clusters` (1,203 rows), `CIP - CC - Sub-Clusters` (4,725), `CC - Sub-Cluster - NAICS` (112), and the combined `SOC - CIP - CC` (6,688). The *Wide*, *Single* and *Interdisciplinary* sheets repeat the same facts in other shapes.
- The combined file has 14 blank-bodied rows: one per cluster, used as section headers. They aren't data.

## 2. What one row is (grain)

**Presumed keys:** SOC file `SOC + sub-cluster`; CIP file `CIP + sub-cluster`; NAICS file `sub-cluster + NAICS`; combined file `SOC + CIP + sub-cluster`.

In [3]:
fullx = full.dropna(subset=["SOC Code"])
print("Combined-file rows that only carry a cluster name (section headers):", full["SOC Code"].isna().sum())
for n, d, key in [("soc", soc, ["SOC Code", "Sub-Cluster"]), ("cip", cip, ["CIP Code (2020)", "Sub-Cluster"]),
                  ("naics", nai, ["Sub-Cluster", "2-Digit NAICS"]), ("full", fullx, ["SOC Code", "CIP Code (2020)", "Sub-Cluster"])]:
    print(f"{n:<6} key {' + '.join(key):<45} duplicates: {d.duplicated(key).sum()}")
print("\nThe 2 exact duplicate rows in the combined file:"); display(fullx[fullx.duplicated(keep=False)])
print("Sub-cluster names reused across clusters:",
      fullx[["Career Cluster", "Sub-Cluster"]].drop_duplicates()["Sub-Cluster"].duplicated().sum())

Combined-file rows that only carry a cluster name (section headers): 14
soc    key SOC Code + Sub-Cluster                        duplicates: 0
cip    key CIP Code (2020) + Sub-Cluster                 duplicates: 0
naics  key Sub-Cluster + 2-Digit NAICS                   duplicates: 0
full   key SOC Code + CIP Code (2020) + Sub-Cluster      duplicates: 2

The 2 exact duplicate rows in the combined file:


,Career Cluster,Sub-Cluster,SOC Code,Occupation Title,CIP Code (2020),CIP Title (2020),2-Digit NAICS,NAICS title
4066,Energy & Natural Resources,Ecological Research & Development,15-2021,Mathematicians,27.0502,Mathematical Statistics and Probability.,"21, 22","Mining, Quarrying, and Oil and Gas Extraction, Utilities"
4067,Energy & Natural Resources,Ecological Research & Development,15-2021,Mathematicians,27.0502,Mathematical Statistics and Probability.,"21, 22","Mining, Quarrying, and Oil and Gas Extraction, Utilities"
6549,Public Service & Safety,"Local, State, & Federal Services",15-2021,Mathematicians,30.4901,Mathematical Economics.,"56, 92","Administrative and Support and Waste Management and Remediation Services, Public Admin..."
6550,Public Service & Safety,"Local, State, & Federal Services",15-2021,Mathematicians,30.4901,Mathematical Economics.,"56, 92","Administrative and Support and Waste Management and Remediation Services, Public Admin..."


Sub-cluster names reused across clusters: 0


**Grain findings**
- SOC file: `SOC + sub-cluster` unique . CIP file: `CIP + sub-cluster` unique . NAICS file: `sub-cluster + NAICS` unique . One row = one link, no counts.
- Combined file: `SOC + CIP + sub-cluster` has 2 exact duplicate rows (Mathematicians × two math CIPs). Harmless; drop them.
- Sub-cluster names are unique across clusters, so the sub-cluster name alone identifies its cluster (a surrogate key is still cleaner).

## 3–4. Profile: clusters, codes, and do the four files agree?

In [4]:
tree = fullx.groupby("Career Cluster").agg(sub_clusters=("Sub-Cluster", "nunique"), jobs=("SOC Code", "nunique"), majors=("CIP Code (2020)", "nunique"))
display(tree)
print(f"{fullx['Career Cluster'].nunique()} clusters, {fullx['Sub-Cluster'].nunique()} sub-clusters")
pairs = lambda d: set(map(tuple, d[["Career Cluster", "Sub-Cluster"]].drop_duplicates().values))
print("Sub-cluster lists differ from the combined file -> SOC:", pairs(soc) ^ pairs(fullx), "| CIP:", pairs(cip) ^ pairs(fullx), "| NAICS:", pairs(nai) ^ pairs(fullx))
trip = lambda d, c: set(map(tuple, d[[c, "Career Cluster", "Sub-Cluster"]].drop_duplicates().values))
print("SOC x sub-cluster, SOC file vs combined: only in one =", len(trip(soc, "SOC Code") ^ trip(fullx, "SOC Code")))
print("CIP x sub-cluster, CIP file vs combined: only in one =", len(trip(cip, "CIP Code (2020)") ^ trip(fullx, "CIP Code (2020)")))
bad = lambda s, pat: sorted(s[~s.str.fullmatch(pat)].unique())
print("Malformed SOC codes:", bad(soc["SOC Code"], r"\d\d-\d{4}"), "| malformed CIP codes:", bad(cip["CIP Code (2020)"], r"\d\d\.\d{4}"))
print("NAICS values, NAICS file:", sorted(nai["2-Digit NAICS"].unique()))
print("NAICS values, combined file:", sorted(fullx["2-Digit NAICS"].unique()))
yn = socw["Interdisciplinary SOC? (Yes/No)"].value_counts().to_dict(); print("SOC wide sheet 'interdisciplinary' flag:", yn)

,sub_clusters,jobs,majors
Career Cluster,,,
Advanced Manufacturing,5,137,214
Agriculture,7,46,173
"Arts, Entertainment, & Design",6,64,149
Construction,4,93,101
Digital Technology,5,20,89
Education,4,83,1332
Energy & Natural Resources,6,87,247
Financial Services,5,40,66
Healthcare & Human Services,6,146,766


14 clusters, 72 sub-clusters
Sub-cluster lists differ from the combined file -> SOC: set() | CIP: set() | NAICS: {('Agriculture', 'water systems')}
SOC x sub-cluster, SOC file vs combined: only in one = 0
CIP x sub-cluster, CIP file vs combined: only in one = 0
Malformed SOC codes: [] | malformed CIP codes: []
NAICS values, NAICS file: ['11', '21', '22', '23', '31-33', '42', '44-45', '48-49', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81', '92']
NAICS values, combined file: ['11', '21, 22', '23', '31-33', '42, 48-49', '51, 54', '52, 53', '54', '55', '56, 92', '61, 92', '62, 81', '71', '72, 44-45']
SOC wide sheet 'interdisciplinary' flag: {'No': 726, 'Yes': 125}


**Profile findings**
- **14 clusters, 72 sub-clusters** (4–7 sub-clusters each), as in the framework guidebook.
- **The four files agree exactly.** Every SOC × sub-cluster and CIP × sub-cluster link in the separate files is in the combined file, and the other way round. The only difference: the NAICS file spells one sub-cluster `water systems` instead of `Water Systems`.
- Codes are clean text: SOC `NN-NNNN`, CIP `NN.NNNN` with leading zeros kept, no malformed values.
- **NAICS is 2-digit sectors only.** The NAICS file has one row per sector; the combined file packs two sectors into one cell (`"21, 22"`, `"62, 81"`), so use the NAICS file.
- 125 of 851 jobs (15%) sit in more than one cluster (Advance CTE's "interdisciplinary" flag).

## 5. Coverage: NYC jobs, openings and degrees

The crosswalk is only useful if it reaches the jobs and majors NYC actually has. Three tests:
1. **SOC 2018 structure** (BLS): which detailed jobs are in the crosswalk, and which codes aren't SOC 2018 at all.
2. **NYC employment** (NYS DOL wages, nb 03) and NYC projected openings 2022-32 (nb 02).
3. NYC 2024 associate's and bachelor's awards, first major, all NYC colleges (IPEDS, nb 05).

In [5]:
st = pd.read_excel(RAW / man.loc[man.key == "bls_soc_structure", "file"].iloc[0], header=7, dtype=str)
det = set(st["Detailed Occupation"].dropna().str.strip())
ttl = dict(zip(st["Detailed Occupation"].dropna().str.strip(), st.loc[st["Detailed Occupation"].notna()].iloc[:, 4].str.strip()))
S = set(soc["SOC Code"])
print(f"SOC 2018 detailed jobs: {len(det)} | crosswalk jobs: {len(S)}")
print("Crosswalk codes that are not SOC 2018 detailed:", sorted(S - det))
miss = sorted(det - S)
print(f"SOC 2018 detailed jobs not in the crosswalk: {len(miss)}")
agg_parent = lambda m_: next((a for a in sorted(S - det) if a[:6] == m_[:6] or (a.endswith("0") and a[:6] == m_[:5] + "0")), "")
cov = pd.DataFrame({"soc": miss, "title": [ttl[m_] for m_ in miss], "covered_by": [agg_parent(m_) for m_ in miss]})
display(cov)
print("SOC 2018 jobs with no aggregate parent in the crosswalk:", cov[cov.covered_by == ""].soc.tolist())

SOC 2018 detailed jobs: 867 | crosswalk jobs: 851
Crosswalk codes that are not SOC 2018 detailed: ['13-1020', '13-2020', '21-1018', '25-2052', '25-9045', '29-2010', '31-1120', '39-7010', '47-4090', '51-2028', '51-2090', '53-1047']
SOC 2018 detailed jobs not in the crosswalk: 28


,soc,title,covered_by
0,13-1021,"Buyers and Purchasing Agents, Farm Products",13-1020
1,13-1022,"Wholesale and Retail Buyers, Except Farm Products",13-1020
2,13-1023,"Purchasing Agents, Except Wholesale, Retail, and Farm Products",13-1020
3,13-2022,Appraisers of Personal and Business Property,13-2020
4,13-2023,Appraisers and Assessors of Real Estate,13-2020
5,21-1011,Substance Abuse and Behavioral Disorder Counselors,21-1018
6,21-1014,Mental Health Counselors,21-1018
7,25-2055,"Special Education Teachers, Kindergarten",25-2052
8,25-2056,"Special Education Teachers, Elementary School",25-2052
9,25-9042,"Teaching Assistants, Preschool, Elementary, Middle, and Secondary School, Except Speci...",25-9045


SOC 2018 jobs with no aggregate parent in the crosswalk: []


In [6]:
W = RAW / man.loc[man.key == "nysdol_oews_regions", "file"].iloc[0]
w = pd.read_csv(W, dtype=str, keep_default_na=False); w.columns = [c.strip() for c in w.columns]
w = w[(w["Area Name"].str.strip() == "New York City") & w["Standard Occupational Code"].str.fullmatch(r"\d\d-\d{4}")
      & ~w["Standard Occupational Code"].str.endswith("0")].set_index("Standard Occupational Code")
emp = pd.to_numeric(w.Employment.str.replace(",", ""), errors="coerce")
print(f"NYC wage jobs (NYS DOL): {len(emp)} codes, {emp.notna().sum()} with employment, {emp.sum():,.0f} jobs")
print(f"  in the crosswalk: {emp.index.isin(S).sum()} codes, {emp[emp.index.isin(S)].sum() / emp.sum():.2%} of employment")
print("  NYC aggregate codes the crosswalk also uses:", sorted(set(emp.index) & (S - det)))

pr = pd.read_csv(RAW / man.loc[man.key == "nysdol_projections", "file"].iloc[0], dtype=str)
pr = pr[(pr.Area == "New York City") & pr.SOC.str.fullmatch(r"\d\d-\d{3}[1-9]")]
op = pd.to_numeric(pr.set_index("SOC")["Total Annual Openings"], errors="coerce")
print(f"\nNYC projections {pr.Period.iloc[0]}: {len(pr)} leaf codes, {op.sum():,.0f} openings a year")
print(f"  in the crosswalk: {pr.SOC.isin(S).sum()} codes, {op[op.index.isin(S)].sum() / op.sum():.3%} of openings")
display(pr[~pr.SOC.isin(S)][["SOC", "Occupation Title", "Total Annual Openings"]])

NYC wage jobs (NYS DOL): 673 codes, 556 with employment, 4,047,620 jobs
  in the crosswalk: 673 codes, 100.00% of employment
  NYC aggregate codes the crosswalk also uses: ['21-1018', '25-2052', '25-9045', '51-2028', '53-1047']

NYC projections 2022 - 2032: 533 leaf codes, 486,963 openings a year
  in the crosswalk: 522 codes, 99.988% of openings


,SOC,Occupation Title,Total Annual Openings
4441,11-9198,"Personal Service Managers; Entertainment & Recreation Managers, Except Gambling; and M...",30
4470,13-2098,"Financial and Investment Analysts, Financial Risk Specialists, and Financial Specialis...",1
4481,15-1256,Software Developers and Software Quality Assurance Analysts and Testers,2
4482,15-1257,Web Developers and Digital Interface Designers,1
4526,19-3031,"Clinical, Counseling, and School Psychologists",3
4618,25-3097,"Teachers and Instructors, All Other, Except Substitute Teachers",3
4698,29-1228,"Physicians, All Other; and Ophthalmologists, Except Pediatric",2
4700,29-1298,"Acupuncturists and Healthcare Diagnosing or Treating Practitioners, All Other",1
4791,39-1098,"First-Line Supervisors of Personal Service & Entertainment and Recreation Workers, Exc...",3
4817,39-9098,"Crematory Operators and Personal Care and Service Workers, All Other",3


In [7]:
def zcsv(key, file_pat):
    f = RAW / man.loc[(man.key == key) & man.file.str.contains(file_pat), "file"].iloc[0]
    z = zipfile.ZipFile(f); n = [n for n in z.namelist() if n.lower().endswith(".csv") and "_rv" not in n.lower()][0]
    d = pd.read_csv(z.open(n), dtype=str, encoding="latin-1"); d.columns = d.columns.str.replace("ï»¿", "").str.strip()
    return d
hd = zcsv("ipeds_directory", "HD2024")
nyc_ids = set(hd[hd.COUNTYCD.isin(["36005", "36047", "36061", "36081", "36085"])].UNITID)
c = zcsv("ipeds_completions", "C2024_A")
c = c[c.UNITID.isin(nyc_ids) & c.AWLEVEL.isin(["3", "5"]) & (c.MAJORNUM == "1") & (c.CIPCODE != "99")].copy()
c["n"] = c.CTOTALT.astype(int); c["level"] = c.AWLEVEL.map({"3": "associate", "5": "bachelor"})
ref = pd.read_csv(RAW / man.loc[man.file.str.endswith("CIPCode2020.csv"), "file"].iloc[0], dtype=str)
ref["code"] = ref.CIPCode.str.strip('="'); six = ref[ref.code.str.fullmatch(r"\d\d\.\d{4}")]
cip_t = dict(zip(six.code, six.CIPTitle.str.strip()))
C = set(cip["CIP Code (2020)"])
print(f"CIP 2020 6-digit programs: {len(six)} | in the crosswalk: {len(C)} | crosswalk codes not in CIP 2020: {sorted(C - set(six.code))}")
for lvl, g in c.groupby("level"):
    m = g[~g.CIPCODE.isin(C)].groupby("CIPCODE").n.sum().sort_values(ascending=False)
    print(f"\n{lvl}: {g.n.sum():,} NYC awards (2024, 1st major); {1 - m.sum() / g.n.sum():.1%} in the crosswalk. Biggest misses:")
    display(pd.DataFrame({"awards": m.head(6), "title": m.head(6).index.map(cip_t)}))

CIP 2020 6-digit programs: 2325 | in the crosswalk: 1946 | crosswalk codes not in CIP 2020: ['99.9999']

associate: 18,490 NYC awards (2024, 1st major); 97.2% in the crosswalk. Biggest misses:


,awards,title
CIPCODE,,
51.0000,449,"Health Services/Allied Health/Health Sciences, General."
11.0899,24,"Computer Software and Media Applications, Other."
44.9999,12,"Public Administration and Social Service Professions, Other."
22.9999,10,"Legal Professions and Studies, Other."
11.1099,9,"Computer/Information Technology Services Administration and Management, Other."
24.0199,7,"Liberal Arts and Sciences, General Studies and Humanities, Other."



bachelor: 54,735 NYC awards (2024, 1st major); 97.1% in the crosswalk. Biggest misses:


,awards,title
CIPCODE,,
51.0000,571,"Health Services/Allied Health/Health Sciences, General."
50.0499,199,"Design and Applied Arts, Other."
52.9999,178,"Business, Management, Marketing, and Related Support Services, Other."
30.0000,140,"Multi-/Interdisciplinary Studies, General."
30.9999,94,"Multi-/Interdisciplinary Studies, Other."
52.0299,77,"Business Administration, Management and Operations, Other."


**Coverage findings**
- **Jobs:** 851 codes. 839 are SOC 2018 detailed jobs; 12 are BLS/OEWS aggregates (`31-1120` home health + personal care aides, `29-2010` lab technologists + technicians, `13-1020` buyers, `21-1018` counselors, `25-2052` special ed teachers, and so on). They stand in for all 28 SOC 2018 jobs that are missing, so nothing is lost; the bridge from nb 06 handles it, as it already does for NYS DOL wages.
- **NYC employment: 100% covered** (all 673 NYS DOL NYC codes, 4.05 M jobs). The crosswalk uses the same aggregate codes NYS DOL does.
- NYC projected openings 2022-32: 99.99% covered. The 11 misses are NYS DOL hybrid codes (`15-1256`, `13-2098` …) with 1–30 openings each; that's leftover detail, since NYC also publishes their parts. Same hybrid bridge as nb 06.
- **Majors:** 1,946 of 2,325 CIP 2020 programs. 97.2% of NYC 2024 associate's awards and 97.1% of bachelor's awards are covered. The biggest misses are general "Health Services/Allied Health, General" `51.0000` (1,020 awards) and the "Other" codes (`50.0499`, `52.9999`, `30.0000` …). These have no SOC link in the NCES crosswalk either (see §6), so they're missing for the same reason.

## 6. Relationships: where do a major's clusters come from?

The combined file lists SOC × CIP pairs. If those pairs are the NCES CIP ↔ SOC crosswalk (nb 06), then a major's clusters are just the clusters of the jobs NCES links it to, and they inherit the D2 problem: almost every major links to a postsecondary teaching job (`25-1xxx`), which sits in Education.

In [8]:
x = pd.read_excel(RAW / man.loc[man.file.str.endswith("CIP2020_SOC2018_Crosswalk.xlsx"), "file"].iloc[0], sheet_name="CIP-SOC", dtype=str).apply(lambda s: s.str.strip())
nces = set(zip(x.CIP2020Code, x.SOC2018Code)); ff = set(zip(fullx["CIP Code (2020)"], fullx["SOC Code"]))
print(f"CIP-SOC pairs: combined file {len(ff):,} | NCES {len(nces):,} | in both {len(ff & nces):,} | only combined {len(ff - nces)} | only NCES {len(nces - ff)}")
only_c = pd.DataFrame(sorted(ff - nces), columns=["cip", "soc"]); only_n = pd.DataFrame(sorted(nces - ff), columns=["cip", "soc"])
print("Only in combined file, by SOC:", only_c.soc.value_counts().head(8).to_dict())
print("Only in NCES, by SOC:", only_n.soc.value_counts().head(8).to_dict())
sk = set(map(tuple, soc[["SOC Code", "Career Cluster", "Sub-Cluster"]].values))
print("Combined rows whose sub-cluster is one of that job's own sub-clusters:",
      f"{fullx.apply(lambda r: (r['SOC Code'], r['Career Cluster'], r['Sub-Cluster']) in sk, axis=1).mean():.1%}")

CIP-SOC pairs: combined file 5,775 | NCES 6,097 | in both 5,705 | only combined 70 | only NCES 392
Only in combined file, by SOC: {'25-2052': 13, '29-2010': 11, '13-2020': 10, '13-1020': 9, '21-1018': 6, '25-9045': 5, '39-7010': 3, '53-1047': 3}
Only in NCES, by SOC: {'99-9999': 194, '11-9121': 69, '25-2055': 13, '25-2056': 13, '13-2022': 9, '13-1022': 7, '25-9043': 7, '29-2012': 7}


Combined rows whose sub-cluster is one of that job's own sub-clusters: 100.0%


In [9]:
f = fullx.rename(columns={"CIP Code (2020)": "cip", "SOC Code": "soc", "Career Cluster": "cluster"})
aw = c.groupby("CIPCODE").n.sum()
d = pd.DataFrame({"all_links": f.groupby("cip").cluster.apply(set),
                  "no_postsec": f[~f.soc.str.startswith("25-1")].groupby("cip").cluster.apply(set)})
d["no_postsec"] = d.no_postsec.apply(lambda s: s if isinstance(s, set) else set())
d["awards"] = d.index.map(aw).fillna(0); T = d.awards.sum()
edu = d.all_links.apply(lambda s: "Education" in s); edu_prof = edu & ~d.no_postsec.apply(lambda s: "Education" in s)
print(f"NYC 2024 awards in majors tagged Education: {d[edu].awards.sum() / T:.1%}; tagged Education ONLY through postsecondary teaching: {d[edu_prof].awards.sum() / T:.1%}")
empty = d.no_postsec.apply(len) == 0
print(f"Majors left with no cluster once 25-1xxx is dropped: {empty.sum()} codes, {d[empty].awards.sum() / T:.1%} of NYC awards")
print(f"Clusters per NYC award: {(d.all_links.apply(len) * d.awards).sum() / T:.2f} with every link, {(d.no_postsec.apply(len) * d.awards).sum() / T:.2f} without 25-1xxx")
top = d.sort_values("awards", ascending=False).head(12)
display(pd.DataFrame({"title": top.index.map(cip_t).str[:45], "awards": top.awards.astype(int),
                      "clusters (Advance CTE)": top.all_links.apply(lambda s: ", ".join(sorted(s))),
                      "without postsecondary teaching": top.no_postsec.apply(lambda s: ", ".join(sorted(s)) or "— none —")}))

NYC 2024 awards in majors tagged Education: 88.5%; tagged Education ONLY through postsecondary teaching: 73.0%
Majors left with no cluster once 25-1xxx is dropped: 94 codes, 9.2% of NYC awards
Clusters per NYC award: 2.88 with every link, 2.15 without 25-1xxx


,title,awards,clusters (Advance CTE),without postsecondary teaching
cip,,,,
42.0101,"Psychology, General.",4602,"Education, Healthcare & Human Services, Management & Entrepreneurship","Healthcare & Human Services, Management & Entrepreneurship"
51.3801,Registered Nursing/Registered Nurse.,3736,"Education, Healthcare & Human Services",Healthcare & Human Services
24.0101,Liberal Arts and Sciences/Liberal Studies.,3439,Education,— none —
52.0201,"Business Administration and Management, Gener",3373,"Advanced Manufacturing, Arts, Entertainment, & Design, Construction, Education, Financ...","Advanced Manufacturing, Arts, Entertainment, & Design, Construction, Financial Service..."
43.0103,Criminal Justice/Law Enforcement Administrati,2047,"Education, Management & Entrepreneurship, Public Service & Safety","Management & Entrepreneurship, Public Service & Safety"
11.0701,Computer Science.,1887,"Digital Technology, Education",Digital Technology
26.0101,"Biology/Biological Sciences, General.",1812,"Agriculture, Education, Energy & Natural Resources, Healthcare & Human Services, Publi...","Agriculture, Education, Energy & Natural Resources, Healthcare & Human Services, Publi..."
52.0301,Accounting.,1643,"Education, Financial Services, Public Service & Safety","Financial Services, Public Service & Safety"
38.0207,Talmudic Studies.,1333,Education,— none —


Relationship findings: this is the key result
- The combined file's CIP ↔ SOC pairs are the NCES crosswalk (5,705 of its 5,775 pairs; the other 70 are NCES pairs recoded to the OEWS aggregates above). NCES pairs that are missing are mostly `99-9999` ("no job match", 194) and `11-9121` Natural Sciences Managers (69; the job is in the SOC file, just not linked to majors).
- Every combined row's sub-cluster is one of that job's sub-clusters (100%). So a major has no cluster of its own: it gets the clusters of the jobs NCES links it to.
- That means majors inherit the D2 problem. Almost every major links to a postsecondary teaching job (`25-1xxx`), and those sit in Education. 88.5% of NYC 2024 awards are in majors tagged Education, and 73.0% get the tag only through college-professor jobs. Liberal Arts, Talmudic Studies and other majors are *only* Education, for that reason. Psychology, Nursing, Computer Science, Accounting and Finance all carry an Education tag because of professor jobs.
- Dropping `25-1xxx` (as D2's degree filter already does for associate's/bachelor's) cuts clusters per award from 2.88 to 2.15, and leaves 94 majors (9.2% of NYC awards) with no cluster, led by Liberal Arts (3,439) and Talmudic Studies (1,333). That matches D2's "transfer pathway / no direct job link" label.
- **Proposed:** derive major → cluster from the D2-filtered major → job links (jobs' clusters), not from the Advance CTE CIP file. Then the umbrellas follow the same rule as everything else on the dashboard. Keep the Advance CTE CIP file for reference and as a check. *(decided later as D6)*

## 7. Tie-out

There are no published counts to tie to: this is a classification, not a dataset of numbers. The checks that stand in for a tie-out are the ones above: the four files agree with each other exactly (§3–4), 14 clusters and 72 sub-clusters match the framework guidebook, and the CIP ↔ SOC pairs are the NCES crosswalk (§6). Below: the NAICS side, set against NYC's 2024 jobs by sector (NYS DOL QCEW, nb 07).

In [10]:
nmap = nai.groupby("2-Digit NAICS")["Career Cluster"].apply(lambda s_: ", ".join(sorted(set(s_))))
qc = pd.read_csv(RAW / man.loc[man.key == "nysdol_qcew_annual", "file"].iloc[0], dtype=str, keep_default_na=False)
qc = qc[(qc.Area == "New York City") & (qc["Area Type"] == "Labor Market Area") & (qc.Year == "2024")]
qc["jobs"] = pd.to_numeric(qc["Average Employment"], errors="coerce")
# nb 07: private sector rows are complete; government is published by ownership and mostly only as a total
priv = qc[(qc.Ownership == "Private") & qc.NAICS.str.fullmatch(r"\d\d") & (qc.NAICS != "00")].copy()
priv["sector"] = priv.NAICS.replace({"31": "31-33", "44": "44-45", "48": "48-49"})
t = priv.groupby("sector").agg(title=("NAICS Title", "first"), nyc_jobs_2024=("jobs", "sum"))
gov = qc[(qc.Ownership == "Total, All Government") & (qc.NAICS == "00")].jobs.sum()
t.loc["gov"] = ["All government (federal, state, local; every industry)", gov]
t["cluster"] = t.index.map(nmap).fillna("— not in crosswalk —")
display(t.sort_values("nyc_jobs_2024", ascending=False))
pt = t.drop("gov").nyc_jobs_2024
print(f"Private NYC jobs: {pt.sum():,.0f}; in a sector the crosswalk maps: {pt[pt.index.isin(nmap.index)].sum() / pt.sum():.2%} (the rest is 99 'Unclassified')")
nk = nai.groupby("2-Digit NAICS")["Career Cluster"].nunique()
print("Sectors in two clusters:", {k_: nmap[k_] for k_ in nk[nk > 1].index})

,title,nyc_jobs_2024,cluster
sector,,,
62,Health Care and Social Assistance,984503,Healthcare & Human Services
gov,"All government (federal, state, local; every industry)",566579,— not in crosswalk —
54,Professional and Technical Services,437038,"Digital Technology, Marketing & Sales"
72,Accommodation and Food Services,354305,"Hospitality, Events, & Tourism"
52,Finance and Insurance,352445,Financial Services
44-45,Retail Trade,294934,"Hospitality, Events, & Tourism"
56,Administrative and Waste Services,240892,Public Service & Safety
51,Information,216299,Digital Technology
61,Educational Services,197363,Education


Private NYC jobs: 4,063,615; in a sector the crosswalk maps: 99.49% (the rest is 99 'Unclassified')
Sectors in two clusters: {'54': 'Digital Technology, Marketing & Sales', '92': 'Education, Public Service & Safety'}


**Tie-out findings**
- No counts to tie to. Internal agreement is exact (§3–4), and the sizes match the guidebook (14 / 72).
- **NAICS:** every private NYC sector maps to a cluster, 99.5% of private jobs (the rest is `99` Unclassified). Each sector has one cluster except 54 Professional & Technical Services (Digital Technology + Marketing & Sales) and 92 Public Administration (Education + Public Service & Safety).
- Some sector placements are surprising and worth labeling: Retail 44-45 → Hospitality, Events & Tourism, Other Services 81 → Healthcare & Human Services, Administrative & Waste 56 → Public Service & Safety. Industry views on the dashboard need their own words ("employers in this sector") per brief S8.
- Government: QCEW publishes NYC government mostly as one total (567k jobs, nb 07). Only `92` is a government sector in the crosswalk; public schools and hospitals belong in `61`/`62` (as in nb 09's bridge).

## 8. Fitness for purpose: one job, several clusters

A job can sit in more than one cluster (Advance CTE flags these "interdisciplinary"). That matters for every cluster total on the dashboard: if a job counts in each of its clusters, cluster totals add up to more than NYC's total. Size of the effect, NYC employment by cluster, two ways.

In [11]:
k = soc.groupby("SOC Code")["Career Cluster"].nunique(); ks = soc.groupby("SOC Code")["Sub-Cluster"].nunique()
print("Jobs by number of clusters:", k.value_counts().sort_index().to_dict())
print("Jobs by number of sub-clusters:", ks.value_counts().sort_index().to_dict())
e = emp.dropna(); e = e[e.index.isin(S)]
print(f"NYC employment in jobs with 2+ clusters: {e[e.index.map(k) > 1].sum() / e.sum():.1%}; with 2+ sub-clusters: {e[e.index.map(ks) > 1].sum() / e.sum():.1%}")
big = e[e.index.map(k) > 1].sort_values(ascending=False).head(10)
display(pd.DataFrame({"title": big.index.map(w["Occupational Title"]).str[:40], "nyc_jobs": big.astype(int),
                      "clusters": [", ".join(sorted(soc.loc[soc["SOC Code"] == s_, "Career Cluster"].unique())) for s_ in big.index]}))

Jobs by number of clusters: {1: 726, 2: 91, 3: 20, 4: 10, 5: 1, 6: 1, 8: 2}
Jobs by number of sub-clusters: {1: 657, 2: 125, 3: 35, 4: 15, 5: 7, 6: 3, 7: 3, 8: 1, 9: 1, 10: 4}
NYC employment in jobs with 2+ clusters: 12.0%; with 2+ sub-clusters: 24.3%


,title,nyc_jobs,clusters
Standard Occupational Code,,,
11-1021,General and Operations Managers,99410,"Energy & Natural Resources, Management & Entrepreneurship"
15-1252,Software Developers,74710,"Arts, Entertainment, & Design, Digital Technology"
13-1111,Management Analysts,44470,"Education, Management & Entrepreneurship, Public Service & Safety"
15-1232,Computer User Support Specialists,18830,"Digital Technology, Healthcare & Human Services"
15-2051,Data Scientists,15960,"Digital Technology, Financial Services, Marketing & Sales"
21-1021,"Child, Family, and School Social Workers",13950,"Education, Healthcare & Human Services"
11-3121,Human Resources Managers,11720,"Management & Entrepreneurship, Public Service & Safety"
15-1255,Web and Digital Interface Designers,11110,"Arts, Entertainment, & Design, Digital Technology"
11-9199,"Managers, All Other",11030,"Arts, Entertainment, & Design, Management & Entrepreneurship, Public Service & Safety"


In [12]:
m = soc[["SOC Code", "Career Cluster"]].drop_duplicates().assign(jobs=lambda t: t["SOC Code"].map(e))
m["share"] = 1 / m.groupby("SOC Code")["Career Cluster"].transform("size")
by = pd.DataFrame({"count_in_every_cluster": m.groupby("Career Cluster").jobs.sum(),
                   "split_equally": (m.jobs * m.share).groupby(m["Career Cluster"]).sum()}).sort_values("count_in_every_cluster", ascending=False)
by.loc["Sum"] = by.sum(); display(by.round(0).astype(int))
print(f"NYC total: {e.sum():,.0f}. Counting a job in every cluster inflates the sum by {by.loc['Sum', 'count_in_every_cluster'] / e.sum() - 1:.1%}.")
FEAT = {"Tech": ["Digital Technology"], "Healthcare": ["Healthcare & Human Services"], "Education": ["Education"],
        "Business": ["Financial Services", "Management & Entrepreneurship", "Marketing & Sales"]}
for name, cl in FEAT.items():
    jobs = set(soc.loc[soc["Career Cluster"].isin(cl), "SOC Code"])
    print(f"Featured '{name}' ({' + '.join(cl)}): {len(jobs)} jobs, {e[e.index.isin(jobs)].sum():,.0f} NYC jobs (each job once)")

,count_in_every_cluster,split_equally
Career Cluster,,
Management & Entrepreneurship,645570,521435
Healthcare & Human Services,573680,525274
"Hospitality, Events, & Tourism",556150,544186
Marketing & Sales,471390,458530
Public Service & Safety,447260,352762
Education,412850,358640
Financial Services,338900,317796
Supply Chain & Transportation,310550,298470
Construction,230690,201917


NYC total: 4,047,620. Counting a job in every cluster inflates the sum by 16.3%.
Featured 'Tech' (Digital Technology): 20 jobs, 212,500 NYC jobs (each job once)
Featured 'Healthcare' (Healthcare & Human Services): 146 jobs, 573,680 NYC jobs (each job once)
Featured 'Education' (Education): 83 jobs, 412,850 NYC jobs (each job once)
Featured 'Business' (Financial Services + Management & Entrepreneurship + Marketing & Sales): 109 jobs, 1,427,790 NYC jobs (each job once)


**Fitness findings**
- 12.0% of NYC employment is in jobs with 2+ clusters (24.3% with 2+ sub-clusters). The big ones: General and Operations Managers (99k: Energy + Management), Software Developers (75k: Digital Technology + Arts, Entertainment & Design), Management Analysts (44k: 3 clusters), Data Scientists (16k: 3).
- Counting a job in every cluster inflates the cluster sum by 16.3% (4.71 M vs 4.05 M). Splitting equally keeps the total but invents fractions (half of each software developer in "Arts"). Advance CTE gives no primary cluster.
- **Proposed:** keep the many-to-many bridge as published. A cluster page counts every job in it, and the dashboard never adds clusters up; a note says "a job can belong to more than one cluster". *(decided later as D6)*
- **Featured areas (brief S7):** Tech = Digital Technology, only 20 jobs (213k NYC jobs). Healthcare & Human Services 146 jobs (574k). Education 83 jobs (413k). "Business" = Financial + Management + Marketing & Sales is 109 jobs and 1.43 M NYC jobs (35% of all), because it includes retail salespersons, cashiers, customer service and office clerks. These clusters hold every job, at any education level; the S10 filter (associate's / bachelor's jobs only) will shrink them. I was going to confirm "Business" after seeing it; this is that view.

## 9. Issues list and proposed staging tables

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Combined file: 14 header rows with only a cluster name; 2 exact duplicate rows | 16 of 6,688 | Drop. Use the separate SOC / CIP / NAICS files instead |
| 2 | 12 SOC codes are BLS aggregates, not SOC 2018 | 12 of 851 jobs | Keep as published; nb 06's aggregate → parts bridge reaches the 28 detailed jobs |
| 3 | NYS DOL hybrid codes (`15-1256`, `13-2098` …) not in the crosswalk | 11 projection codes, 0.01% of openings | Same hybrid bridge as nb 06 |
| 4 | A major's clusters come from CIP → SOC links, so they carry the D2 professor problem (73% of NYC awards tagged Education only through `25-1xxx`) | most CIPs | **Candidate D6:** derive major → cluster from D2-filtered major → job links; keep the CIP file for reference |
| 5 | Jobs in several clusters; no primary cluster | 125 jobs, 12% of NYC employment | **Candidate D6:** many-to-many bridge; never sum across clusters on the dashboard |
| 6 | `water systems` lower case in the NAICS file | 1 row | Fix case in staging |
| 7 | NAICS only at 2-digit sectors; 54 and 92 sit in two clusters | 2 sectors | Fine for v1 (nb 07 also chose 2-digit sectors). Show both clusters |
| 8 | "Other" and "General" CIPs not covered (`51.0000`, `50.0499`, `52.9999`, `30.0000` …) | ~2.9% of NYC awards | Show as "no cluster"; same reason they have no job link |
| 9 | No version number in the files | all | Record the download date and file hashes (manifest); check the page yearly |

**Proposed staging tables** (Advance CTE = `cte`):
- `stg_cte__cluster`: one row per sub-cluster: `cluster_name`, `subcluster_name`, surrogate ids. 72 rows.
- `stg_cte__soc_subcluster`: `soc_code` (SOC 2018 or BLS aggregate), `subcluster_id`. 1,203 rows. Grain: job × sub-cluster.
- `stg_cte__naics_subcluster`: `naics_sector` (`31-33` style), `subcluster_id`. 112 rows.
- `stg_cte__cip_subcluster`: `cip_code` (CIP 2020), `subcluster_id`. 4,725 rows, reference only if D6 is agreed.

In [13]:
print(f"stg_cte__cluster:            {fullx[['Career Cluster', 'Sub-Cluster']].drop_duplicates().shape[0]} rows (cluster, sub-cluster)")
print(f"stg_cte__soc_subcluster:     {len(soc):,} rows, {soc['SOC Code'].nunique()} jobs")
print(f"stg_cte__cip_subcluster:     {len(cip):,} rows, {cip['CIP Code (2020)'].nunique()} majors (kept for reference, see issue 4)")
print(f"stg_cte__naics_subcluster:   {len(nai)} rows, {nai['2-Digit NAICS'].nunique()} sectors")

stg_cte__cluster:            72 rows (cluster, sub-cluster)
stg_cte__soc_subcluster:     1,203 rows, 851 jobs
stg_cte__cip_subcluster:     4,725 rows, 1946 majors (kept for reference, see issue 4)
stg_cte__naics_subcluster:   112 rows, 20 sectors
